# <span style="color:darkblue"> Project 1 and 2: Web Scraping Information about Projects Funded by Adaptation Fund </span>

**Course:** DATASCI 530 - Computing I  
**Date:** October 22, 2025  

**Data Source:** [Adaptation Fund Projects Database](https://www.adaptation-fund.org/projects-programmes/project-information/projects-table-view/) 

## <span style="color:darkblue"> Executive Summary </span>

**About Adaptation Fund:**
- The Adaptation Fund is an international climate finance mechanism that funds concrete adaptation projects in developing countries vulnerable to climate change
- Established under the Kyoto Protocol and became operational in 2010
- Projects focus on practical, on-the-ground adaptation solutions in sectors like agriculture, water management, coastal protection, and disaster risk reduction
- Emphasizes direct access - allowing developing countries to receive funding directly through national implementing entities
- Projects span multiple regions including Africa, Asia-Pacific, Latin America, and Small Island Developing States


**Objectives of the analysis**

This analysis examines climate adaptation projects funded by the Adaptation Fund to understand:
- Distribution of funding across countries and regions
- Sectoral focus of climate adaptation initiatives
- Project implementation timelines and status
- Key implementing entities and partnerships

**Key Details:**

- **Total Projects/Entries Scraped:** All adaptation projects listed on the website (181 projects)
- **Data Collection Date:** October 22, 2025
- **Geographic Coverage:** Multiple countries across Africa, Asia, Latin America, and other regions
- **Project Types:** Completed projects, projects under implementation, and approved proposals

## <span style="color:darkblue"> Data Collection Methodology </span>

<font size = 5>Overview of Web Scraping Approach
<font size = 3>
- Set table display to 100 entries per page (maximum available)
- Navigate through 2 pages to capture all 181 projects
- Click into each project detail page to extract comprehensive information
- Extract "At a Glance" metadata for each project

<font size = 5> Automated Navigation Process
<font size = 3>
1. **Initialize browser** and navigate to projects table
2. **Modify dropdown** to show 100 entries per page
3. **Loop through each project:**
   - Click project link
   - Extract project name from header
   - Extract all "At a Glance" fields dynamically
   - Navigate back to table
   - Move to next project
4. **Paginate** to second page and repeat extraction
5. **Compile data** into structured dataset

<font size = 5> 
Data Fields Collected
<font size = 3>

The fields available within the "At a Glance" section varied based on project status.

The following fields were available and collected for all projects:
- **Project Name:** Full title of the adaptation project
- **Country/Region:** Geographic location and regional classification
- **Sector:** Primary sector(s) of intervention
- **Grant Amount:** Total funding approved (USD)
- **Implementing Entity:** Organization responsible for project implementation
- **Executing Entity:** On-ground execution partners
- **Approval Date:** Date of project approval
- **Duration:** Expected project timeline
- **Status:** Current project status (Completed, Under Implementation, Proposal Approved)

Additional data fields were available and collected for projects that are completed and under implementation
- **Start date:** Actual start date of project
- **Expected completion date:** Expected completion date of project, as listed on proposal and based on when project actually started

Some data fields existed only for completed projects
- **Revised completion date:** Actual completion date of the project
- **Location:** Google maps link to the location of the project (note: this will not be used in analysis)
- **Project ID:** A sequence to help identify completed projects

## <span style="color:darkblue"> Technical Implementation </span>

### Import packages and libraries

In [59]:
#import scripts
exec(open("./scripts/import_packages.py").read())
from selenium.webdriver.common.keys import Keys

### Step 1: Initialize Web Driver and Navigate to Website

In [60]:
# Open browser to start web scraping
opts = Options()
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=opts)

# Navigate to specific website
starting_url = 'https://www.adaptation-fund.org/projects-programmes/project-information/projects-table-view/'
driver.get(starting_url)

### Step 2: Test Extraction on First Project

**Purpose:** 
- Verify scraping logic works on a single project before scaling
- Understand data structure and field availability
- Debug any issues with element identification

**Testing Process:**
1. Extract first row from table
2. Click into project detail page
3. Check if the clicking worked
4. Extract project name and "At a Glance" fields
5. Examine extracted data structure
6. Navigate back to verify navigation works

**Testing process (steps 1 to 3)**

In [63]:
# Get the first row
all_rows = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
first_row = all_rows[0]

# Find the link in the first cell (the project name cell)
project_link = first_row.find_element('xpath', './/td[1]/a')

# Click the link
project_link.send_keys(Keys.RETURN)

# Wait for the detail page to load
time.sleep(3)

# Print the current URL to confirm we're on the detail page
print(f"Current URL: {driver.current_url}")

Current URL: https://www.adaptation-fund.org/project/transforming-communities-a-nexus-of-climate-smart-agriculture-livelihood-diversification-and-womens-economic-empowerment/


**Testing process (steps 4 and 5)**

In [ ]:
# Extract project name from the page header
project_name = driver.find_element('xpath', '//h1[@class="entry-title"]').text
print(f"Project name: {project_name}\n")

# Find all the info boxes within "At a Glance"
info_boxes = driver.find_elements('xpath', '//div[@class="project-info-box"]')

print(f"Found {len(info_boxes)} info boxes")

# Create empty dictionary for this project
project_data = {'project_name': project_name}

# Loop through each info box
for box in info_boxes:
    # Extract label (h4)
    label = box.find_element('xpath', './/h4').text
    
    # Extract value from project-terms div
    value = box.find_element('xpath', './/div[@class="project-terms"]').text
    
    # Clean the label
    clean_label = label.replace(":", "").replace("/", "_").replace(" ", "_").lower()
    
    # Store in dictionary
    project_data[clean_label] = value

# Print the extracted data
print(f"\nExtracted {len(project_data)} fields:")
for key, val in project_data.items():
    print(f"{key}: {val}")

Project name: Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment

Found 8 info boxes

Extracted 9 fields:
project_name: Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment
country_region: Ethiopia/Africa
sector: Multi-sector
grant_amount: USD 9,999,328
implementing_entity: Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia
executing_entity: Ministry of Water and Energy,Ministry of Agriculture
approval_date: 10/10/2025
duration: 3 years
status: Proposal Approved


**Testing process (step 6 - navigate back to main page)**

In [ ]:
# Go back to the main table page
driver.back()

# Wait for the table to load
time.sleep(2)

# Verify we're back by checking if the table exists
all_rows = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
print(f"\nBack on main page. Number of rows: {len(all_rows)}")


Back on main page. Number of rows: 100


### Step 3: Configure Table Display
- Modify dropdown to show 100 entries per page
- Reduces pagination from 19 pages to 2 pages

In [20]:
# Find the dropdown element (the select tag, not the label)
dropdown = driver.find_element('xpath', '//select[@name="projects-table_length"]')

# Convert to a Select object
from selenium.webdriver.support.ui import Select
select_dropdown = Select(dropdown)

# Select the option for 100 entries
select_dropdown.select_by_value('100')

# Wait for the page to reload
import time
time.sleep(3)

# Now count how many rows we have
all_rows_100 = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
print(f"Number of rows after selecting: {len(all_rows_100)}")

Number of rows after selecting: 100


### Step 4: Extract Data from Projects in Page 1

**Process:**
- Loop through all 100 projects on first page
- Click each project link to access detail page
- Dynamically extract all available "At a Glance" fields
- Handle varying field structures across different project statuses
- Navigate back and continue to next project

In [ ]:
# Find all rows on page 1
all_rows = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
print(f"Starting to scrape {len(all_rows)} projects from Page 1...\n")

# Create list to store all project data
all_projects_data = []

# Loop through each row on page 1
for i, row in enumerate(all_rows):
    print(f"Scraping project {i+1}/{len(all_rows)} (Page 1)...")
    
    # Find and click the project link
    project_link = row.find_element('xpath', './/td[1]/a')
    project_link.send_keys(Keys.RETURN)
    time.sleep(2)
    
    # Extract project name
    project_name = driver.find_element('xpath', '//h1[@class="entry-title"]').text
    
    # Extract "At a Glance" info
    info_boxes = driver.find_elements('xpath', '//div[@class="project-info-box"]')
    project_data = {'project_name': project_name}
    
    for box in info_boxes:
        label = box.find_element('xpath', './/h4').text
        value = box.find_element('xpath', './/div[@class="project-terms"]').text
        clean_label = label.replace(":", "").replace("/", "_").replace(" ", "_").lower()
        project_data[clean_label] = value
    
    # Store this project
    all_projects_data.append(project_data)
    
    # Go back to main table
    driver.back()
    time.sleep(2)
    
    # Re-find all rows (page refreshes after going back)
    all_rows = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')

print(f"\n Page 1 complete! Scraped {len(all_projects_data)} projects")

Starting to scrape 100 projects from Page 1...

Scraping project 1/100 (Page 1)...
Scraping project 2/100 (Page 1)...
Scraping project 3/100 (Page 1)...
Scraping project 4/100 (Page 1)...
Scraping project 5/100 (Page 1)...
Scraping project 6/100 (Page 1)...
Scraping project 7/100 (Page 1)...
Scraping project 8/100 (Page 1)...
Scraping project 9/100 (Page 1)...
Scraping project 10/100 (Page 1)...
Scraping project 11/100 (Page 1)...
Scraping project 12/100 (Page 1)...
Scraping project 13/100 (Page 1)...
Scraping project 14/100 (Page 1)...
Scraping project 15/100 (Page 1)...
Scraping project 16/100 (Page 1)...
Scraping project 17/100 (Page 1)...
Scraping project 18/100 (Page 1)...
Scraping project 19/100 (Page 1)...
Scraping project 20/100 (Page 1)...
Scraping project 21/100 (Page 1)...
Scraping project 22/100 (Page 1)...
Scraping project 23/100 (Page 1)...
Scraping project 24/100 (Page 1)...
Scraping project 25/100 (Page 1)...
Scraping project 26/100 (Page 1)...
Scraping project 27/100 (

### Step 5: Navigate to Page 2 and Extract Remaining Projects

**Process:**
- Click "Next" button to navigate to page 2
- Repeat extraction process for remaining projects (~81 projects)
- Append to same data list for combined dataset

In [ ]:
# Click the Next button to go to page 2
next_button = driver.find_element('xpath', '//*[@id="projects-table_next"]')
next_button.click()
time.sleep(3)

In [25]:
# Find all rows on page 2
all_rows_page2 = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')
print(f"Starting to scrape {len(all_rows_page2)} projects from Page 2...\n")

# Loop through each row on page 2
for i, row in enumerate(all_rows_page2):
    print(f"Scraping project {i+1}/{len(all_rows_page2)} (Page 2)...")
    
    # Find and click the project link
    project_link = row.find_element('xpath', './/td[1]/a')
    project_link.send_keys(Keys.RETURN)
    time.sleep(5)
    
    # Extract project name
    project_name = driver.find_element('xpath', '//h1[@class="entry-title"]').text
    
    # Extract "At a Glance" info
    info_boxes = driver.find_elements('xpath', '//div[@class="project-info-box"]')
    project_data = {'project_name': project_name}
    
    for box in info_boxes:
        label = box.find_element('xpath', './/h4').text
        value = box.find_element('xpath', './/div[@class="project-terms"]').text
        clean_label = label.replace(":", "").replace("/", "_").replace(" ", "_").lower()
        project_data[clean_label] = value
    
    # Store this project
    all_projects_data.append(project_data)
    
    # Go back to main table
    driver.back()
    time.sleep(2)
    
    # Re-find all rows
    all_rows_page2 = driver.find_elements('xpath', '//*[@id="projects-table"]/tbody/tr')

print(f"\nPage 2 complete! Scraped {len(all_projects_data) - 100} additional projects")
print(f"\n{'='*60}")
print(f"DATA COLLECTION COMPLETE")
print(f"Total projects scraped: {len(all_projects_data)}")
print(f"{'='*60}\n")

Starting to scrape 81 projects from Page 2...

Scraping project 1/81 (Page 2)...
Scraping project 2/81 (Page 2)...
Scraping project 3/81 (Page 2)...
Scraping project 4/81 (Page 2)...
Scraping project 5/81 (Page 2)...
Scraping project 6/81 (Page 2)...
Scraping project 7/81 (Page 2)...
Scraping project 8/81 (Page 2)...
Scraping project 9/81 (Page 2)...
Scraping project 10/81 (Page 2)...
Scraping project 11/81 (Page 2)...
Scraping project 12/81 (Page 2)...
Scraping project 13/81 (Page 2)...
Scraping project 14/81 (Page 2)...
Scraping project 15/81 (Page 2)...
Scraping project 16/81 (Page 2)...
Scraping project 17/81 (Page 2)...
Scraping project 18/81 (Page 2)...
Scraping project 19/81 (Page 2)...
Scraping project 20/81 (Page 2)...
Scraping project 21/81 (Page 2)...
Scraping project 22/81 (Page 2)...
Scraping project 23/81 (Page 2)...
Scraping project 24/81 (Page 2)...
Scraping project 25/81 (Page 2)...
Scraping project 26/81 (Page 2)...
Scraping project 27/81 (Page 2)...
Scraping project 

### Step 6: Convert to DataFrame and Initial Data Inspection

**Process:**
- Convert list of dictionaries to Pandas DataFrame
- Check dataset dimensions
- Verify data structure
- Display sample of collected data

In [26]:
# Convert to DataFrame
adaptation_projects = pd.DataFrame(all_projects_data)

#save raw dataframe as CSV
adaptation_projects.to_csv('data_raw/adaptation_fund_projects.csv', index=False)

In [58]:
# Display basic information

print("DATASET OVERVIEW")
print("-"*60)
print(f"\nDataFrame shape: {adaptation_projects.shape}")
print(f"Number of projects: {adaptation_projects.shape[0]}")
print(f"Number of fields: {adaptation_projects.shape[1]}")

print(f"\nColumn names:")
for i, col in enumerate(adaptation_projects.columns, 1):
    print(f"  {i}. {col}")

print("\n" + "-"*60)

DATASET OVERVIEW
------------------------------------------------------------

DataFrame shape: (181, 18)
Number of projects: 181
Number of fields: 18

Column names:
  1. project_name
  2. country_region
  3. sector
  4. grant_amount
  5. implementing_entity
  6. executing_entity
  7. approval_date
  8. duration
  9. status
  10. transferred_amount
  11. start_date
  12. expected_completion_date
  13. locations
  14. revised_completion_date
  15. completion_date
  16. project_id
  17. grant_amount_clean
  18. duration_clean

------------------------------------------------------------


In [32]:
# Display first few rows
print("Sample of collected data (first 3 projects):")
adaptation_projects.head(3).style

Sample of collected data (first 3 projects):


,project_name,country_region,sector,grant_amount,implementing_entity,executing_entity,approval_date,duration,status,transferred_amount,start_date,expected_completion_date,locations,revised_completion_date,completion_date,project_id
0,"Transforming Communities: A Nexus of Climate-Smart Agriculture, Livelihood Diversification, and Women’s Economic Empowerment",Ethiopia/Africa,Multi-sector,"USD 9,999,328",Ministry of Finance and Economic Cooperation of the Federal Democratic Republic of Ethiopia,"Ministry of Water and Energy,Ministry of Agriculture",10/10/2025,3 years,Proposal Approved,nan,nan,nan,nan,nan,nan,nan
1,Sustainable Pasture Management and Adaptation with Resilient Technologies for Herders in Mongolia (SMART-Herders),Mongolia/Asia-Pacific,Agriculture,"USD 2,038,883",International Fund Agricultural Dev,Office of the President (Government of Mongolia) United Nations Industrial Development Organization (UNIDO),04/11/2025,4 years,Proposal Approved,"USD 852,494",nan,nan,nan,nan,nan,nan
2,Improving adaptive capacity of vulnerable and food-insecure populations in Lesotho Phase II (IACoV-2),Lesotho/Africa,Food Security,"USD 10,000,000",UN World Food Programme,"Ministry of Environment and Forestry,Ministry of Agriculture, Food Security and Nutrition",04/11/2025,5 years,Proposal Approved,"USD 2,928,280",nan,nan,nan,nan,nan,nan


## <span style="color:darkblue"> Results </span>

### (a) Quality Checks

**Objective:** Verify data completeness and integrity
- Count total observations collected
- Check for empty/null values in dataset
- Ensure data extraction was successful

In [34]:
# Count total observations
total_projects = len(adaptation_projects)
print(f"Total number of observations: {total_projects}")

# Check for completely empty rows
empty_rows = adaptation_projects.isnull().all(axis=1).sum()
print(f"Completely empty rows: {empty_rows}")

# Check if any critical fields are completely empty
critical_fields = ['project_name', 'country_region', 'grant_amount', 'status']
print("\nChecking critical fields:")

for field in critical_fields:
    if field in adaptation_projects.columns:
        non_empty = adaptation_projects[field].notna().sum()
        print(f"  {field}: {non_empty}/{total_projects} non-empty ({non_empty/total_projects*100:.1f}%)")
    else:
        print(f"  {field}: Field not found in dataset")

Total number of observations: 181
Completely empty rows: 0

Checking critical fields:
  project_name: 181/181 non-empty (100.0%)
  country_region: 181/181 non-empty (100.0%)
  grant_amount: 181/181 non-empty (100.0%)
  status: 181/181 non-empty (100.0%)


### (b) Mean/Variance for numeric fields

**Objective:** Calculate mean and variance for grant amount and duration of projects 

Process:
- Clean up columns to remove strings (e.g, "USD" or "years")
- calculate mean and variance

**Clean up 'grant_amount' column**

In [56]:
# Remove "USD" prefix
adaptation_projects['grant_amount_clean'] = adaptation_projects['grant_amount'].str.replace('USD', '')

# Remove commas
adaptation_projects['grant_amount_clean'] = adaptation_projects['grant_amount_clean'].str.replace(',', '')

# Remove any extra spaces
adaptation_projects['grant_amount_clean'] = adaptation_projects['grant_amount_clean'].str.strip()

# Convert to numeric
adaptation_projects['grant_amount_clean'] = pd.to_numeric(adaptation_projects['grant_amount_clean'])

# Check the result
#print("Cleaned grant_amount values:")
#print(adaptation_projects['grant_amount_clean'].head(2))

**Clean up 'duration' column**

In [55]:
# Check what duration looks like
#print("Sample duration values:")
#print(adaptation_projects['duration'].head(2))

# Remove " years" text
adaptation_projects['duration_clean'] = adaptation_projects['duration'].str.replace(' years', '')

# Remove " year" (for singular, if any)
adaptation_projects['duration_clean'] = adaptation_projects['duration_clean'].str.replace(' year', '')

# Remove any extra spaces
adaptation_projects['duration_clean'] = adaptation_projects['duration_clean'].str.strip()

# Convert to numeric
adaptation_projects['duration_clean'] = pd.to_numeric(adaptation_projects['duration_clean'])

# Check the result
#print("Cleaned duration values:")
#print(adaptation_projects['duration_clean'].head(2))

**Calculate mean and variance**

In [54]:
# Grant Amount statistics
grant_mean = adaptation_projects['grant_amount_clean'].mean()
grant_var = adaptation_projects['grant_amount_clean'].var()

print("GRANT AMOUNT STATISTICS")
print(f"Average grant amount: ${grant_mean:,.2f}")
print(f"Variance: {grant_var:,.2f}")
print(f"Std Dev: ${adaptation_projects['grant_amount_clean'].std():,.2f}")
print(f"Min Grant: ${adaptation_projects['grant_amount_clean'].min():,.2f}")
print(f"Max Grant: ${adaptation_projects['grant_amount_clean'].max():,.2f}")

print("\n" + "-"*50)
print("PROJECT DURATION STATISTICS")
duration_mean = adaptation_projects['duration_clean'].mean()
duration_var = adaptation_projects['duration_clean'].var()

print(f"Mean: {duration_mean:.2f} years")
print(f"Variance: {duration_var:.2f}")
print(f"Min duration: {adaptation_projects['duration_clean'].min():,} year")
print(f"Max duration: {adaptation_projects['duration_clean'].max():,} years")


GRANT AMOUNT STATISTICS
Average grant amount: $7,358,645.06
Variance: 12,742,940,384,339.52
Std Dev: $3,569,725.53
Min Grant: $689,264.00
Max Grant: $14,000,000.00

--------------------------------------------------
PROJECT DURATION STATISTICS
Mean: 4.22 years
Variance: 0.86
Min duration: 1.0 year
Max duration: 6.0 years


## <span style="color:darkblue"> Discussion </span>


Successfully collected comprehensive data on 181 climate adaptation projects funded by the Adaptation Fund across multiple countries and regions. The web scraping process demonstrates the effectiveness of automated data collection from interactive websites with dynamic content.

### Notable findings from initial analysis:
- **Funding scale varies significantly:** Grant amounts range from $689,264 to $14 million, with an average of $7.36 million per project
- **Standard project duration:** Projects typically last 4.2 years on average; but projects can be as short as 1 year
- **Data completeness challenges:** Some fields have missing values due to different project statuses (Completed, Under Implementation, Proposal Approved), which have varying data requirements
- **Field variability:** Completed projects contain additional fields (start date, completion dates) not present in approved proposals, requiring dynamic extraction approach

### Limitations and Considerations

- **Missing data patterns:** Fields like start_date, completion_date, and revised_completion_date are only available for projects beyond the proposal stage
- **Data format inconsistencies:** Required cleaning of currency values and duration fields to enable quantitative analysis
    - I'll be cleaning other columns in project 2 (e.g, splitting country and region from 'country/region' column) 
- **Time for extraction of data:** Both pages took me around 10-15 minutes to extract data from. This is mainly because I had to set sleep time in between clicking on link and extracting information to 5 seconds. If I had it any shorter, I got errors since the page hadn't refreshed on time

### Future Analysis (Project 2)

In Project 2, this dataset will be analyzed in depth to answer the following research questions:

**Geographic and Sectoral Distribution:**
- Which countries and regions receive the most adaptation funding?
- What sectors (agriculture, water, coastal, multi-sector) are prioritized?
- Are there patterns in funding allocation across different climate vulnerability zones?

**Temporal Patterns:**
- How has project types evolved over time?
- What is the relationship between project duration and grant amount?

**Implementation Analysis:**
- Which implementing entities are most active in climate adaptation?
- What is the disbursement rate (transferred vs. approved amounts)?

This foundational dataset provides a robust basis for understanding global climate adaptation investment patterns and implementation strategies.